In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D,
    Conv1D, BatchNormalization, Activation, RepeatVector,
    TimeDistributed, Attention, Concatenate, AdditiveAttention,Attention,Lambda, Add)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import layers, Model, Input

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

import numpy as np, re
import pandas as pd
import matplotlib.pyplot as plt
import gc, os, time
from keras import backend as K
from vmdpy import VMD

Yt = pd.read_csv('ws.csv', header=1, parse_dates=['Timestamp'])
Yt = Yt.rename(columns={'Timestamp': 'time'})

Yt['wind_sin'] = np.sin(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))
Yt['wind_cos'] = np.cos(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))

# 2. 湍流强度 SD（四层）轻度 clip
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s']

def clip_sd(col):
    Q1 = Yt[col].quantile(0.25)
    Q3 = Yt[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    Yt[col] = np.clip(Yt[col], None, upper)

for c in sd_cols:
    clip_sd(c)

# 3. 湍流强度 TI = SD / Mean
Yt['TI_110'] = Yt['Ch1_Anem_110.00m_E_SD_m/s'] / Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['TI_50']  = Yt['Ch2_Anem_50.00m_E_SD_m/s']  / Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['TI_30']  = Yt['Ch3_Anem_30.00m_E_SD_m/s']  / Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['TI_10']  = Yt['Ch4_Anem_10.00m_E_SD_m/s']  / Yt['Ch4_Anem_10.00m_E_Avg_m/s']

Yt.replace([np.inf, -np.inf], np.nan, inplace=True)
Yt.fillna(0, inplace=True)

# 4. 阵风偏差（gust deviation）
Yt['gust_dev_110'] = Yt['Ch1_Anem_110.00m_E_Gust_m/s'] - Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['gust_dev_50']  = Yt['Ch2_Anem_50.00m_E_Gust_m/s']  - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['gust_dev_30']  = Yt['Ch3_Anem_30.00m_E_Gust_m/s']  - Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['gust_dev_10']  = Yt['Ch4_Anem_10.00m_E_Gust_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# 5. 层间风切变（vertical shear）
Yt['shear_110_10'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_110_50'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['shear_50_10']  = Yt['Ch2_Anem_50.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_30_10']  = Yt['Ch3_Anem_30.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# 6. 气象变量：温度 + 气压
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# 7. 时间特征（分钟周期）
Yt['minute'] = Yt['time'].dt.minute
Yt['minute_sin'] = np.sin(2 * np.pi * Yt['minute'] / 60)
Yt['minute_cos'] = np.cos(2 * np.pi * Yt['minute'] / 60)

features = [

    # 原始风速
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',

    # 风向
    'wind_sin', 'wind_cos',

    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',

    # TI
    'TI_110', 'TI_50', 'TI_30', 'TI_10',

    # gust deviation
    'gust_dev_110', 'gust_dev_50', 'gust_dev_30', 'gust_dev_10',

    # shear
    'shear_110_10', 'shear_110_50', 'shear_50_10', 'shear_30_10',

    # 气象变量
    'temp', 'pressure',

    # 时间周期
    'minute_sin', 'minute_cos']

targets = [
    'Ch1_Anem_110.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch4_Anem_10.00m_E_Avg_m/s']

K_vmd = 8
alpha_vmd = 3500

def vmd_comp(df, col, K=K_vmd, alpha=alpha_vmd):
    signal = df[col].values
    signal = signal - np.mean(signal)
    tau = 0
    DC = 0
    init = 1
    tol = 1e-7
    u, u_hat, omega = VMD(signal, alpha, tau, K, DC, init, tol)
    for k in range(K):
        imf = u[k]
        if len(imf) < len(df):
            imf = np.append(imf, imf[-1])
        df[f"{col}_IMF{k+1}"] = imf
    return df

target_h = ["Ch1_Anem_110.00m_E_Avg_m/s"]
for th in target_h:
    print("Decomposing:", th)
    Yt = vmd_comp(Yt, col=th, K=K_vmd, alpha=alpha_vmd)
    for k in range(1, K_vmd + 1):
        imf_col = f"{th}_IMF{k}"
        if imf_col not in features:
            features.append(imf_col)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D,
    Conv1D, BatchNormalization, Activation, RepeatVector,
    TimeDistributed, Attention, Concatenate, AdditiveAttention,
    Lambda, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import layers, Model, Input

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import numpy as np, re
import pandas as pd
import matplotlib.pyplot as plt
import gc, os, time
from keras import backend as K
from vmdpy import VMD


Yt = pd.read_csv(
    'ws.csv',
    header=1,
    parse_dates=['Timestamp']
)

Yt = Yt.rename(
    columns={'Timestamp': 'time'}
)


Yt['wind_sin'] = np.sin(
    np.deg2rad(
        Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360
    )
)

Yt['wind_cos'] = np.cos(
    np.deg2rad(
        Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360
    )
)


# 2. 湍流强度 SD（四层）轻度 clip
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s'
]


def clip_sd(col):
    Q1 = Yt[col].quantile(0.25)
    Q3 = Yt[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    Yt[col] = np.clip(
        Yt[col],
        None,
        upper
    )


for c in sd_cols:
    clip_sd(c)

# 3. 湍流强度 TI = SD / Mean
Yt['TI_110'] = (
    Yt['Ch1_Anem_110.00m_E_SD_m/s']
    / Yt['Ch1_Anem_110.00m_E_Avg_m/s'])

Yt['TI_50'] = (
    Yt['Ch2_Anem_50.00m_E_SD_m/s']
    / Yt['Ch2_Anem_50.00m_E_Avg_m/s'])

Yt['TI_30'] = (
    Yt['Ch3_Anem_30.00m_E_SD_m/s']
    / Yt['Ch3_Anem_30.00m_E_Avg_m/s'])

Yt['TI_10'] = (
    Yt['Ch4_Anem_10.00m_E_SD_m/s']
    / Yt['Ch4_Anem_10.00m_E_Avg_m/s'])

Yt.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True)

Yt.fillna(
    0,
    inplace=True)

# 4. 阵风偏差（gust deviation）
Yt['gust_dev_110'] = (
    Yt['Ch1_Anem_110.00m_E_Gust_m/s']
    - Yt['Ch1_Anem_110.00m_E_Avg_m/s'])

Yt['gust_dev_50'] = (
    Yt['Ch2_Anem_50.00m_E_Gust_m/s']
    - Yt['Ch2_Anem_50.00m_E_Avg_m/s'])

Yt['gust_dev_30'] = (
    Yt['Ch3_Anem_30.00m_E_Gust_m/s']
    - Yt['Ch3_Anem_30.00m_E_Avg_m/s'])

Yt['gust_dev_10'] = (
    Yt['Ch4_Anem_10.00m_E_Gust_m/s']
    - Yt['Ch4_Anem_10.00m_E_Avg_m/s'])

# 5. 层间风切变（vertical shear）
Yt['shear_110_10'] = (
    Yt['Ch1_Anem_110.00m_E_Avg_m/s']
    - Yt['Ch4_Anem_10.00m_E_Avg_m/s'])

Yt['shear_110_50'] = (
    Yt['Ch1_Anem_110.00m_E_Avg_m/s']
    - Yt['Ch2_Anem_50.00m_E_Avg_m/s'])

Yt['shear_50_10'] = (
    Yt['Ch2_Anem_50.00m_E_Avg_m/s']
    - Yt['Ch4_Anem_10.00m_E_Avg_m/s'])

Yt['shear_30_10'] = (
    Yt['Ch3_Anem_30.00m_E_Avg_m/s']
    - Yt['Ch4_Anem_10.00m_E_Avg_m/s'])

# 6. 气象变量：温度 + 气压
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# 7. 时间特征（分钟周期）
Yt['minute'] = Yt['time'].dt.minute

Yt['minute_sin'] = np.sin(
    2 * np.pi * Yt['minute'] / 60)

Yt['minute_cos'] = np.cos(
    2 * np.pi * Yt['minute'] / 60)

features = [

    # 原始风速
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',

    # 风向
    'wind_sin',
    'wind_cos',

    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',

    # TI
    'TI_110',
    'TI_50',
    'TI_30',
    'TI_10',

    # gust deviation
    'gust_dev_110',
    'gust_dev_50',
    'gust_dev_30',
    'gust_dev_10',

    # shear
    'shear_110_10',
    'shear_110_50',
    'shear_50_10',
    'shear_30_10',

    # 气象变量
    'temp',
    'pressure',

    # 时间周期
    'minute_sin',
    'minute_cos']

targets = [
    'Ch1_Anem_110.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch4_Anem_10.00m_E_Avg_m/s']

K_vmd = 8
alpha_vmd = 3500

def vmd_comp(df, col, K=K_vmd, alpha=alpha_vmd):
    signal = df[col].values
    signal = signal - np.mean(signal)

    tau = 0
    DC = 0
    init = 1
    tol = 1e-7

    u, u_hat, omega = VMD(signal,alpha,tau,K,DC,init,tol)

    for k in range(K):
        imf = u[k]
        if len(imf) < len(df):
            imf = np.pad(
                imf,
                (0, len(df) - len(imf)),
                mode='edge')
        elif len(imf) > len(df):
            imf = imf[:len(df)]
        df[f"{col}_IMF{k+1}"] = imf
    return df

target_h = "Ch1_Anem_110.00m_E_Avg_m/s"
df_y = Yt[['time'] + features].copy()
df_y = df_y.sort_values('time').reset_index(drop=True)
split_idx = int(0.8 * len(df_y))

train_raw = df_y.iloc[:split_idx].copy().reset_index(drop=True)
test_raw = df_y.iloc[split_idx:].copy().reset_index(drop=True)
print("Decomposing train:",target_h)

train_vmd = vmd_comp(
    train_raw,
    col=target_h,
    K=K_vmd,
    alpha=alpha_vmd)

print("Decomposing test:",target_h)

test_vmd = vmd_comp(
    test_raw,
    col=target_h,
    K=K_vmd,
    alpha=alpha_vmd)

vmd_features = features.copy()

for k in range(1,K_vmd + 1):
    imf_col = f"{target_h}_IMF{k}"
    if imf_col not in vmd_features:
        vmd_features.append(imf_col)

np.random.seed(42)
tf.random.set_seed(42)

def dataset_multi(data_scaled,y_raw,window_size=3,pred_steps=1):
    X, y = [], []
    n = len(data_scaled)
    for i in range(
        n - window_size - pred_steps + 1):
        X.append(
            data_scaled[
                i:i + window_size,
                :
            ]
        )

        y.append(
            y_raw[
                i + window_size:
                i + window_size + pred_steps
            ]
        )

    return np.array(X), np.array(y)


def metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_val = mean_absolute_error(y_true, y_pred)
    r2_val = r2_score(y_true, y_pred)
    mape_val = np.mean(
        np.abs(
            (y_true - y_pred) /
            np.maximum(np.abs(y_true), 1e-6)
        )
    ) * 100

    return rmse_val, mae_val, mape_val, r2_val

w_list = [3, 6, 12, 24]
f_list = [1, 2, 3, 4, 5, 6]

results = []
predictions = {}

target_list = [
    "Ch1_Anem_110.00m_E_Avg_m/s"]

for target in target_list:
    train_df = train_vmd[vmd_features].copy()
    test_df = test_vmd[vmd_features].copy()

    y_train_raw = train_vmd[target].values.astype(float)
    y_test_raw = test_vmd[target].values.astype(float)

    scaler_X = StandardScaler()
    train_scaled = scaler_X.fit_transform(train_df.values)
    test_scaled = scaler_X.transform(test_df.values)

    for w in w_list:
        for pred_steps in f_list:
            forecast_min = pred_steps * 10
            print(
                f"\nTraining {target}: "
                f"w={w}, "
                f"Predict next {forecast_min} minutes")

            X_train, y_train = dataset_multi(
                train_scaled,
                y_train_raw,
                window_size=w,
                pred_steps=pred_steps)

            X_test, y_test = dataset_multi(
                test_scaled,
                y_test_raw,
                window_size=w,
                pred_steps=pred_steps)
            
            scaler_y = MinMaxScaler()
            y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).reshape(-1,pred_steps)
            y_test_s = scaler_y.transform(y_test.reshape(-1, 1)).reshape(-1,pred_steps)
            inp = Input(shape=(w,X_train.shape[2]))
            x = Bidirectional(
                LSTM(
                    64,
                    return_sequences=False,
                    activation='tanh',
                    recurrent_activation='sigmoid'))(inp)

            x = Dense(64,activation='relu')(x)
            out = Dense(pred_steps,activation='linear')(x)
            model = Model(inp,out)
            model.compile(optimizer='adam',loss='mse')

            start_time = time.time()
            history = model.fit(
                X_train,
                y_train_s,
                epochs=150,
                batch_size=128,
                validation_split=0.1,
                shuffle=False,
                callbacks=[
                    ReduceLROnPlateau(
                        monitor='val_loss',
                        factor=0.5,
                        patience=5,
                        min_lr=1e-5,
                        verbose=1),
                    EarlyStopping(
                        monitor='val_loss',
                        patience=10,
                        restore_best_weights=True,
                        verbose=1)],
                verbose=0)
            
            elapsed = (time.time() - start_time)
            print(f"Training time: "f"{elapsed:.2f} sec")

            y_train_pred_s = model.predict(
                X_train,verbose=0)

            y_test_pred_s = model.predict(
                X_test,verbose=0)

            y_train_pred = scaler_y.inverse_transform(
                y_train_pred_s.reshape(-1, 1)).reshape(-1,pred_steps)

            y_test_pred = scaler_y.inverse_transform(
                y_test_pred_s.reshape(-1, 1)).reshape(-1,pred_steps)

            y_train_true = y_train
            y_test_true = y_test

            y_train_true_f = y_train_true.flatten()
            y_train_pred_f = y_train_pred.flatten()

            y_test_true_f = y_test_true.flatten()
            y_test_pred_f = y_test_pred.flatten()


            rmse_train, mae_train, mape_train, r2_train = metrics(
                y_train_true_f,
                y_train_pred_f)

            rmse_test, mae_test, mape_test, r2_test = metrics(
                y_test_true_f,
                y_test_pred_f)

            results.append({
                "Height": target,
                "Window": w,
                "Forecast_Steps": pred_steps,
                "Forecast_Min": forecast_min,
                "Train_RMSE": float(rmse_train),
                "Test_RMSE": float(rmse_test),
                "Train_MAE": float(mae_train),
                "Test_MAE": float(mae_test),
                "Train_MAPE": float(mape_train),
                "Test_MAPE": float(mape_test),
                "Train_R2": float(r2_train),
                "Test_R2": float(r2_test),
                "Time_sec": float(elapsed)})

            print(
                f"Train RMSE={rmse_train:.6f}, "
                f"Test RMSE={rmse_test:.6f}")

            print(
                f"Train MAE={mae_train:.6f}, "
                f"Test MAE={mae_test:.6f}")

            print(
                f"Train MAPE={mape_train:.2f}%, "
                f"Test MAPE={mape_test:.2f}%")

            print(
                f"Train R2={r2_train:.6f}, "
                f"Test R2={r2_test:.6f}")

            predictions[
                (
                    w,
                    pred_steps,
                    target)] = {
                "train_true":
                    y_train_true_f.copy(),

                "train_pred":
                    y_train_pred_f.copy(),

                "test_true":
                    y_test_true_f.copy(),

                "test_pred":
                    y_test_pred_f.copy(),

                "unit": "m/s"}

            K.clear_session()
            gc.collect()

df_results = pd.DataFrame(
    results)
df_results = df_results.sort_values(
    [
        "Forecast_Min",
        "Window"]).reset_index(
    drop=True)

display(df_results)

In [ ]:
base_dir = "5-110m"
os.makedirs(base_dir, exist_ok=True)
safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target_h[0])
for (w, f, target), data in predictions.items():
    safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target)
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)
    df_test = pd.DataFrame({
        "True_Test": data["test_true"],
        "Pred_Test": data["test_pred"]
    })
    filename = f"{safe_h}_test_{f*10}min.csv"
    df_test.to_csv(os.path.join(w_dir, filename), index=False)

df_results = pd.DataFrame(results)
for w in df_results["Window"].unique():
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)
    df_w = df_results[df_results["Window"] == w]
    df_w.to_csv(os.path.join(w_dir, f"summary_w{w}.csv"), index=False)

colors = ["#006699", "#b30000", "#009933",
          "#ff9900", "#660066", "#666600"]

plt.rcParams["font.size"] = 13
def plot_saved_by_w(predictions_dict, base_dir):
    for idx, ((w, f, target), data) in enumerate(predictions_dict.items()):
        w_dir = os.path.join(base_dir, f"w{w}")
        os.makedirs(w_dir, exist_ok=True)

        true_vals = data["test_true"]
        pred_vals = data["test_pred"]

        rmse = np.sqrt(np.mean((true_vals - pred_vals) ** 2))
        r2 = r2_score(true_vals, pred_vals)

        forecast_min = f * 10

        min_val = min(true_vals.min(), pred_vals.min())
        max_val = max(true_vals.max(), pred_vals.max())

        plt.figure(figsize=(7, 7))
        plt.scatter(
            true_vals,
            pred_vals,
            alpha=0.35,
            color=colors[idx % len(colors)],
            edgecolor="none"
        )

        plt.plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Ideal Fit"
        )

        plt.xlabel("True Wind Speed (m/s)")
        plt.ylabel("Predicted Wind Speed (m/s)")
        plt.title(f"{forecast_min}-Minute Ahead Prediction (w={w})")

        plt.text(
            min_val,
            max_val,
            f"$R^2 = {r2:.4f}$\nRMSE = {rmse:.4f}",
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.85)
        )

        plt.grid(alpha=0.35)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            os.path.join(w_dir, f"scatter_{forecast_min}min.png"),
            dpi=300
        )
        plt.close()

plot_saved_by_w(predictions, base_dir="5-110m")